# import

In [1]:
import pandas as pd
import numpy as np
import cv2 as cv
from pathlib import Path 
import seaborn as sns
import matplotlib.pyplot as plt
from StatTools.generators.ndfnoise_generator import ndfnoise
from tqdm import tqdm
import plotly.express as px

In [2]:
def draw_traj(trajectories, frame_shape:tuple, thickness:int):
    ants_num = trajectories.shape[1]
    bgs = [np.zeros(shape=frame_shape, dtype=np.uint8) for _ in range(ants_num)]
    trajs = []
    for ant in range(ants_num):
        trajs.append(np.array([np.array(traj) for traj in trajectories[:, ant]]))
    for ant, bg in zip(trajs, bgs):
        cv.polylines(bg, [ant], isClosed=False, color=(255), thickness=thickness)
    
    return bgs


def calc_iou(tracks, thickness,  frame_shape):
    bgs = draw_traj(tracks, frame_shape=frame_shape, thickness = thickness)
    bgs_arr_bool = np.stack(bgs).astype(np.bool)
    union = bgs_arr_bool.sum(axis=0)
    union_mask = union.astype(np.bool)
    atleast_two_track_intersection = union - union_mask
    iou = atleast_two_track_intersection.sum() / union.sum()
    return iou, atleast_two_track_intersection, union


def gen_traj(frame_num: int, ants_num: int, frame_shape: tuple, margin:int = 10,
             hurst_move: float = 0.5, hurst_species: float = 0.5, 
             start_point = None):
    

    dx = ndfnoise(shape=(frame_num, ants_num), hurst=[hurst_move, hurst_species], normalize=True, dtype=np.float32)
    dy = ndfnoise(shape=(frame_num, ants_num), hurst=[hurst_move, hurst_species], normalize=True, dtype=np.float32)

    x = dx.round().cumsum(axis=0).astype(np.int32)
    y = dy.round().cumsum(axis=0).astype(np.int32)

    if start_point is None:
        start_x = np.random.randint(margin, frame_shape[1] - margin, (ants_num,))
        start_y = np.random.randint(margin, frame_shape[0] - margin, (ants_num,))
        start_point = np.stack([start_x, start_y], axis=1)

    trajectories = np.stack([x, y], axis=2)
    trajectories = trajectories + start_point

    return trajectories

# synth monte-carlo


In [3]:
frame_num = 5000
result = []
seed_list = np.arange(100)
hurst_list = [0.8, 0.9, 1.0, 1.1, 1.2]
lens = np.exp(np.arange(np.log(10), np.log(frame_num), 0.6)).astype(np.int32)
pbar = tqdm(total=len(seed_list)*len(hurst_list)*len(lens))
for seed in seed_list:
    np.random.seed(seed)
    for hurst_move in hurst_list:
        

        config = {
        'frame_num' : frame_num,
        'frame_shape' : (500, 500),
        'margin': 50,
        'hurst_move': hurst_move,
        'hurst_species': 0.8,
        'ants_num' : 1000,
        'start_point' : None
        }
        
        trajectories = gen_traj(**config)

        lens = np.exp(np.arange(np.log(10), np.log(config['frame_num']), 0.6)).astype(np.int32)
        
        for length in lens:
            track_slice = trajectories[:length]
            iou, _, _  = calc_iou(track_slice, 10, config['frame_shape'])

            result.append({'hurst_move':hurst_move,
                           'seed':seed,
                            'length':length,
                            'iou': iou,
                        })
            
            pbar.update(1)      

df = pd.DataFrame(result)

  0%|          | 0/5500 [00:00<?, ?it/s]/home/akhiyarov/.local/lib/python3.12/site-packages/StatTools/generators/ndfnoise_generator.py:51: RuntimeWarning: divide by zero encountered in power
  S *= np.abs(f.reshape(reshape)) ** (-(hurst[i] - 0.5))
/home/akhiyarov/.local/lib/python3.12/site-packages/StatTools/generators/ndfnoise_generator.py:55: RuntimeWarning: divide by zero encountered in power
  S *= np.abs(f_last.reshape((1,) * (dim - 1) + (-1,))) ** (-(hurst[-1] - 0.5))
100%|██████████| 5500/5500 [34:14<00:00,  1.49it/s] 

In [ ]:
# df.to_pickle('montecarlo_hurst_iou_eval.pkl')

In [5]:
px.violin(df.sort_values(['hurst_move', 'length']), x='length', y='iou', color='hurst_move', height=500, width=1000)

# long eval

In [53]:
long_traj = pd.read_csv('gen_ants_gt.csv')

In [54]:
x_wide = long_traj.pivot(columns='ant_id', index='frame', values='x')
y_wide = long_traj.pivot(columns='ant_id', index='frame', values='y')
trajectories = np.stack([x_wide.values, y_wide.values], axis=-1)

In [57]:
lens = np.exp(np.arange(np.log(10), np.log(5000), 0.4)).astype(np.int32)
_result = [] 
for length in lens:
    track_slice = trajectories[:length]
    iou, _, _  = calc_iou(track_slice, 10, (500, 500))

    _result.append({
                    'length':length,
                    'iou': iou,
                })

In [59]:
px.line(_result, x='length', y='iou', height=500, width=1000)